# LLaDA -> Text Projector -> Qwen Training Notebook

This notebook mirrors `pts/train/train.py` in DART, with a small-data option enabled by default.

Workflow:
1. Install dependencies in the notebook environment
2. Configure training (default here: 500 train samples total)
3. Build the mixed dataset and generate LLaDA draft plans
4. Cap training records to a small set for fast iteration
5. Attach and train `model.text_projector` on Qwen answer-token cross-entropy
6. Save projector checkpoints and run quick evaluation

In [1]:
import subprocess
import sys

packages = [
    'transformers==4.49.0',
    'datasets',
    'accelerate',
    'sentencepiece',
    'protobuf',
    'huggingface_hub',
    'tqdm',
    'numpy'
 ]

cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + packages
print('Installing dependencies...')
subprocess.run(cmd, check=True)
print('Dependency installation completed.')

Installing dependencies...


KeyboardInterrupt: 

In [ ]:
print('Standalone mode enabled: no DART repo, no pts imports, no project clone.')

In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Models
ANSWER_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
LLADA_MODEL_ID = 'GSAI-ML/LLaDA-8B-Instruct'

# Small-data training mode
TOTAL_TRAIN_SAMPLES = 500
PER_DATASET_TRAIN_SAMPLES = 72  # 72 * 7 = 504 before exact cap to TOTAL_TRAIN_SAMPLES
MAX_TEST_SAMPLES = 100
MAX_LENGTH = 512

# Generation lengths
DRAFT_PLAN_MAX_NEW_TOKENS = 96
ANSWER_MAX_NEW_TOKENS = 64

ENABLE_OOM_FALLBACK = True
LIGHTLY_TUNE_QWEN = False

# Projector tuning
PROJECTOR_BOTTLENECK_DIM = 512
PROJECTOR_DROPOUT = 0.10
PROJECTOR_LEARNING_RATE = 1e-4

# Optimization
NUM_TRAIN_EPOCHS = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 10
SAVE_STEPS = 100

OUTPUT_ROOT = Path('/kaggle/working/dart_text_projector')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_DIR = OUTPUT_ROOT / 'final_merged'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Output root: {OUTPUT_ROOT}')
print(f'Answer model: {ANSWER_MODEL_ID}')
print(f'LLaDA model: {LLADA_MODEL_ID}')
print(f'TOTAL_TRAIN_SAMPLES={TOTAL_TRAIN_SAMPLES} (pre-cap pool from 7 datasets: {PER_DATASET_TRAIN_SAMPLES * 7})')
print(f'Projector: bottleneck={PROJECTOR_BOTTLENECK_DIM}, dropout={PROJECTOR_DROPOUT}, lr={PROJECTOR_LEARNING_RATE}')

In [ ]:
import gc
import re
from typing import Dict, List, Tuple

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed
from transformers.models.llama.modeling_llama import LlamaRMSNorm
from torch import nn

set_seed(SEED)

ARC_QUESTION_PROMPT_TEMPLATE = """Question: {question}\n{choices_text}"""
ARC_QUESTION_POSTFIX = (
    "\nAnswer with a single letter (A, B, C, or D) and no explanation. "
    "Your answer should start with \"Answer: \" and be followed by the letter "
    "of the answer you choose. Do not include any other text in your response."
)

def free_cuda_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def llada_plan_prompt(question: str) -> str:
    return (
        'You are a careful planning assistant.\n'
        'Given the math question, produce only a concise step-by-step PLAN.\n'
        'Do not output the final numeric answer.\n\n'
        f'Question: {question}\n\n'
        'Plan:'
    )

def build_qwen_prompt_parts(question: str) -> Tuple[str, str]:
    prefix = (
        'You are a math solver.\n'
        'Use the refined plan to solve the question.\n'
        'Return only the final numeric answer.\n\n'
        f'Question: {question}\n\n'
        'Refined plan:\n'
    )
    suffix = '\n\nAnswer:'
    return prefix, suffix

def extract_last_number(text: str) -> str:
    nums = re.findall(r'[-+]?\d+[\d,]*(?:\.\d+)?', text)
    return nums[-1].replace(',', '') if nums else ''

def normalize_answer_text(answer) -> str:
    if answer is None:
        return ''
    return str(answer).strip().replace(',', '')

def prepare_arc_sample(item: Dict) -> Dict[str, str]:
    question = item['question']
    choices = item['choices']
    choices_text = '\n'.join([f"{label}. {text}" for label, text in zip(choices['label'], choices['text'])])
    input_text = ARC_QUESTION_PROMPT_TEMPLATE.format(question=question, choices_text=choices_text) + ARC_QUESTION_POSTFIX
    answer_key = normalize_answer_text(item['answerKey'])
    return {'question': input_text, 'gold_final': answer_key}

def prepare_dart_sample(item: Dict) -> Dict[str, str]:
    question = item['query']
    answer_key = normalize_answer_text(item['gt_ans'])
    return {'question': f'Question: {question}', 'gold_final': answer_key}

def _uniform_sample_indices(total: int, sample_count: int, rng: random.Random) -> List[int]:
    if total <= 0:
        return []
    if total >= sample_count:
        return rng.sample(range(total), sample_count)
    return [rng.randrange(total) for _ in range(sample_count)]

def build_uniform_mixed_train_set(per_dataset_samples: int, seed: int) -> List[Dict[str, str]]:
    rng = random.Random(seed)
    mixed_records: List[Dict[str, str]] = []

    dataset_defs: List[Tuple[str, str, str, int]] = [
        ('arc_easy', 'allenai/ai2_arc', 'ARC-Easy', 0),
        ('arc_challenge', 'allenai/ai2_arc', 'ARC-Challenge', 0),
        ('dart_1', 'hkust-nlp/dart-math-pool-math', '', 1),
        ('dart_2', 'hkust-nlp/dart-math-pool-math', '', 2),
        ('dart_3', 'hkust-nlp/dart-math-pool-math', '', 3),
        ('dart_4', 'hkust-nlp/dart-math-pool-math', '', 4),
        ('dart_5', 'hkust-nlp/dart-math-pool-math', '', 5),
    ]

    for name, dataset_id, config_name, dart_level in dataset_defs:
        if name.startswith('arc'):
            ds = load_dataset(dataset_id, config_name, split='train')
            indices = _uniform_sample_indices(len(ds), per_dataset_samples, rng)
            for idx in indices:
                mixed_records.append(prepare_arc_sample(ds[int(idx)]))
        else:
            ds = load_dataset(dataset_id, split='train')
            ds = ds.filter(lambda x: x['query_metadata']['level'] == dart_level)
            indices = _uniform_sample_indices(len(ds), per_dataset_samples, rng)
            for idx in indices:
                mixed_records.append(prepare_dart_sample(ds[int(idx)]))

    rng.shuffle(mixed_records)
    return mixed_records

def build_draft_generator():
    tokenizer = AutoTokenizer.from_pretrained(LLADA_MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        LLADA_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto' if torch.cuda.is_available() else None,
    )

    if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
    if hasattr(model, 'generation_config') and model.generation_config is not None:
        model.generation_config.use_cache = False

    return pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        do_sample=False,
        max_new_tokens=DRAFT_PLAN_MAX_NEW_TOKENS,
    )

def generate_draft_plan(question: str, draft_generator) -> str:
    prompt = llada_plan_prompt(question)
    try:
        out = draft_generator(prompt, return_full_text=False, use_cache=False)[0]['generated_text']
        out = out.strip()
        return out if out else 'Parse the quantities and solve step-by-step.'
    except torch.cuda.OutOfMemoryError:
        if not ENABLE_OOM_FALLBACK:
            raise
        free_cuda_memory()
        return 'Parse the quantities and solve step-by-step.'

def add_text_projector(model, config):
    model.bottleneck_dim = 1024
    hidden_size = config.hidden_size
    model.text_projector = nn.Sequential(
        nn.Linear(hidden_size, model.bottleneck_dim),
        nn.GELU(approximate='tanh'),
        nn.Linear(model.bottleneck_dim, model.bottleneck_dim),
        nn.GELU(approximate='tanh'),
        nn.Linear(model.bottleneck_dim, hidden_size),
        LlamaRMSNorm(hidden_size, eps=config.rms_norm_eps),
    ).to(next(model.parameters()).device)
    return model

def init_text_projector(model) -> None:
    with torch.no_grad():
        nn.init.xavier_uniform_(model.text_projector[0].weight)
        nn.init.zeros_(model.text_projector[0].bias)
        nn.init.xavier_uniform_(model.text_projector[2].weight)
        nn.init.zeros_(model.text_projector[2].bias)
        nn.init.xavier_uniform_(model.text_projector[4].weight)
        nn.init.zeros_(model.text_projector[4].bias)

class PlanProjectorTrainer(nn.Module):
    def __init__(self, lightly_tune_qwen: bool = False):
        super().__init__()
        use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        self.model_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

        self.tokenizer = AutoTokenizer.from_pretrained(ANSWER_MODEL_ID, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.qwen = AutoModelForCausalLM.from_pretrained(
            ANSWER_MODEL_ID,
            trust_remote_code=True,
            torch_dtype=self.model_dtype,
        )

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.qwen = self.qwen.to(self.device)
        self.qwen = add_text_projector(self.qwen, self.qwen.config)
        init_text_projector(self.qwen)
        self.qwen.text_projector = self.qwen.text_projector.to(device=self.device, dtype=self.model_dtype)

        self._configure_trainable_params(lightly_tune_qwen=lightly_tune_qwen)

    def _configure_trainable_params(self, lightly_tune_qwen: bool) -> None:
        for param in self.qwen.parameters():
            param.requires_grad = False

        for param in self.qwen.text_projector.parameters():
            param.requires_grad = True

        if lightly_tune_qwen:
            if hasattr(self.qwen, 'lm_head'):
                for param in self.qwen.lm_head.parameters():
                    param.requires_grad = True
            if hasattr(self.qwen, 'model') and hasattr(self.qwen.model, 'norm'):
                for param in self.qwen.model.norm.parameters():
                    param.requires_grad = True

    def trainable_parameters(self):
        return [p for p in self.qwen.parameters() if p.requires_grad]

    def _tokenize_no_special(self, text: str, max_len: int) -> torch.Tensor:
        ids = self.tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_len,
            return_tensors='pt',
        )['input_ids']
        return ids.to(self.device)

    def compute_loss(self, question: str, draft_plan: str, gold_final: str, max_length: int) -> torch.Tensor:
        prefix, suffix = build_qwen_prompt_parts(question)
        target = gold_final.strip() + self.tokenizer.eos_token

        prefix_ids = self._tokenize_no_special(prefix, max_length)
        draft_ids = self._tokenize_no_special(draft_plan, max_length)
        suffix_ids = self._tokenize_no_special(suffix, max_length)
        target_ids = self._tokenize_no_special(target, max_length)

        max_total = max_length
        keep_target = min(target_ids.shape[1], max_total)
        target_ids = target_ids[:, -keep_target:]

        available_context = max_total - keep_target
        if available_context < 1:
            available_context = 1

        context_ids = torch.cat([prefix_ids, draft_ids, suffix_ids], dim=1)
        if context_ids.shape[1] > available_context:
            context_ids = context_ids[:, -available_context:]

        prefix_len = min(prefix_ids.shape[1], context_ids.shape[1])
        draft_len = min(draft_ids.shape[1], max(context_ids.shape[1] - prefix_len, 0))

        embeddings = self.qwen.get_input_embeddings()
        context_embeds = embeddings(context_ids)

        if draft_len > 0:
            draft_start = prefix_len
            draft_end = min(draft_start + draft_len, context_embeds.shape[1])
            draft_embeds = context_embeds[:, draft_start:draft_end, :]
            projected = self.qwen.text_projector(draft_embeds.to(self.model_dtype)).to(context_embeds.dtype)
            context_embeds[:, draft_start:draft_end, :] = projected

        target_embeds = embeddings(target_ids)
        inputs_embeds = torch.cat([context_embeds, target_embeds], dim=1)

        attention_mask = torch.ones(inputs_embeds.shape[:2], device=self.device, dtype=torch.long)
        labels = torch.full((1, inputs_embeds.shape[1]), -100, device=self.device, dtype=torch.long)
        labels[:, context_embeds.shape[1]:] = target_ids

        outputs = self.qwen(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)
        return outputs.loss

    @torch.no_grad()
    def generate_answer(self, question: str, draft_plan: str, max_length: int, max_new_tokens: int) -> str:
        self.qwen.eval()
        prefix, suffix = build_qwen_prompt_parts(question)

        prefix_ids = self._tokenize_no_special(prefix, max_length)
        draft_ids = self._tokenize_no_special(draft_plan, max_length)
        suffix_ids = self._tokenize_no_special(suffix, max_length)

        context_ids = torch.cat([prefix_ids, draft_ids, suffix_ids], dim=1)
        if context_ids.shape[1] > max_length:
            context_ids = context_ids[:, -max_length:]

        prefix_len = min(prefix_ids.shape[1], context_ids.shape[1])
        draft_len = min(draft_ids.shape[1], max(context_ids.shape[1] - prefix_len, 0))

        embeddings = self.qwen.get_input_embeddings()
        context_embeds = embeddings(context_ids)

        if draft_len > 0:
            draft_start = prefix_len
            draft_end = min(draft_start + draft_len, context_embeds.shape[1])
            draft_embeds = context_embeds[:, draft_start:draft_end, :]
            projected = self.qwen.text_projector(draft_embeds.to(self.model_dtype)).to(context_embeds.dtype)
            context_embeds[:, draft_start:draft_end, :] = projected

        output_ids = self.qwen.generate(
            inputs_embeds=context_embeds,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        gen_only = output_ids[:, context_embeds.shape[1]:]
        return self.tokenizer.decode(gen_only[0], skip_special_tokens=True).strip()

print('Building 7-dataset uniform train set...')
train_seed_records = build_uniform_mixed_train_set(PER_DATASET_TRAIN_SAMPLES, SEED)
eval_seed_records = train_seed_records[: min(MAX_TEST_SAMPLES, len(train_seed_records))] if MAX_TEST_SAMPLES > 0 else []
print(f'Train samples: {len(train_seed_records)} | Eval samples: {len(eval_seed_records)}')

print('Loading LLaDA draft generator...')
draft_generator = build_draft_generator()

train_records: List[Dict[str, str]] = []
eval_records: List[Dict[str, str]] = []

print('Generating LLaDA draft plans...')
for ex in train_seed_records:
    draft_plan = generate_draft_plan(ex['question'], draft_generator)
    train_records.append({
        'question': ex['question'],
        'draft_plan': draft_plan,
        'gold_final': ex['gold_final'],
    })

for ex in eval_seed_records:
    draft_plan = generate_draft_plan(ex['question'], draft_generator)
    eval_records.append({
        'question': ex['question'],
        'draft_plan': draft_plan,
        'gold_final': ex['gold_final'],
    })

del draft_generator
free_cuda_memory()

print('Loading Qwen + text_projector...')
model_wrapper = PlanProjectorTrainer(lightly_tune_qwen=LIGHTLY_TUNE_QWEN)
trainable_count = sum(p.numel() for p in model_wrapper.trainable_parameters())
print(f'Trainable parameters: {trainable_count}')

In [ ]:
# Rebuild projector with small-data-friendly settings.
hidden_size = model_wrapper.qwen.config.hidden_size
device = model_wrapper.device
dtype = model_wrapper.model_dtype

tuned_projector = nn.Sequential(
    nn.Linear(hidden_size, PROJECTOR_BOTTLENECK_DIM),
    nn.GELU(approximate='tanh'),
    nn.Dropout(PROJECTOR_DROPOUT),
    nn.Linear(PROJECTOR_BOTTLENECK_DIM, PROJECTOR_BOTTLENECK_DIM),
    nn.GELU(approximate='tanh'),
    nn.Dropout(PROJECTOR_DROPOUT),
    nn.Linear(PROJECTOR_BOTTLENECK_DIM, hidden_size),
    LlamaRMSNorm(hidden_size, eps=model_wrapper.qwen.config.rms_norm_eps),
).to(device=device, dtype=dtype)

with torch.no_grad():
    nn.init.xavier_uniform_(tuned_projector[0].weight)
    nn.init.zeros_(tuned_projector[0].bias)
    nn.init.xavier_uniform_(tuned_projector[3].weight)
    nn.init.zeros_(tuned_projector[3].bias)
    nn.init.xavier_uniform_(tuned_projector[6].weight)
    nn.init.zeros_(tuned_projector[6].bias)

model_wrapper.qwen.text_projector = tuned_projector
model_wrapper._configure_trainable_params(lightly_tune_qwen=LIGHTLY_TUNE_QWEN)

trainable_count = sum(p.numel() for p in model_wrapper.trainable_parameters())
print('Applied tuned projector settings.')
print(f'Projector bottleneck={PROJECTOR_BOTTLENECK_DIM}, dropout={PROJECTOR_DROPOUT}')
print(f'Trainable parameters after projector update: {trainable_count}')

In [ ]:
# Exact small-data cap: train on 500 total samples.
if len(train_records) > TOTAL_TRAIN_SAMPLES:
    random.shuffle(train_records)
    train_records = train_records[:TOTAL_TRAIN_SAMPLES]

# Keep eval small as well.
if len(eval_records) > MAX_TEST_SAMPLES:
    eval_records = eval_records[:MAX_TEST_SAMPLES]

print(f'After cap -> train: {len(train_records)} | eval: {len(eval_records)}')

In [ ]:
# Patch PlanProjectorTrainer methods to avoid in-place tensor writes that break autograd.
def _compute_loss_no_inplace(self, question: str, draft_plan: str, gold_final: str, max_length: int) -> torch.Tensor:
    prefix, suffix = build_qwen_prompt_parts(question)
    target = gold_final.strip() + self.tokenizer.eos_token

    prefix_ids = self._tokenize_no_special(prefix, max_length)
    draft_ids = self._tokenize_no_special(draft_plan, max_length)
    suffix_ids = self._tokenize_no_special(suffix, max_length)
    target_ids = self._tokenize_no_special(target, max_length)

    max_total = max_length
    keep_target = min(target_ids.shape[1], max_total)
    target_ids = target_ids[:, -keep_target:]

    available_context = max_total - keep_target
    if available_context < 1:
        available_context = 1

    context_ids = torch.cat([prefix_ids, draft_ids, suffix_ids], dim=1)
    if context_ids.shape[1] > available_context:
        context_ids = context_ids[:, -available_context:]

    prefix_len = min(prefix_ids.shape[1], context_ids.shape[1])
    draft_len = min(draft_ids.shape[1], max(context_ids.shape[1] - prefix_len, 0))

    embeddings = self.qwen.get_input_embeddings()
    context_embeds = embeddings(context_ids)

    if draft_len > 0:
        draft_start = prefix_len
        draft_end = min(draft_start + draft_len, context_embeds.shape[1])
        draft_embeds = context_embeds[:, draft_start:draft_end, :]
        projected = self.qwen.text_projector(draft_embeds.to(self.model_dtype)).to(context_embeds.dtype)

        before = context_embeds[:, :draft_start, :]
        after = context_embeds[:, draft_end:, :]
        context_embeds = torch.cat([before, projected, after], dim=1)

    target_embeds = embeddings(target_ids)
    inputs_embeds = torch.cat([context_embeds, target_embeds], dim=1)

    attention_mask = torch.ones(inputs_embeds.shape[:2], device=self.device, dtype=torch.long)
    labels = torch.full((1, inputs_embeds.shape[1]), -100, device=self.device, dtype=torch.long)
    labels[:, context_embeds.shape[1]:] = target_ids

    outputs = self.qwen(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)
    return outputs.loss

@torch.no_grad()
def _generate_answer_no_inplace(self, question: str, draft_plan: str, max_length: int, max_new_tokens: int) -> str:
    self.qwen.eval()
    prefix, suffix = build_qwen_prompt_parts(question)

    prefix_ids = self._tokenize_no_special(prefix, max_length)
    draft_ids = self._tokenize_no_special(draft_plan, max_length)
    suffix_ids = self._tokenize_no_special(suffix, max_length)

    context_ids = torch.cat([prefix_ids, draft_ids, suffix_ids], dim=1)
    if context_ids.shape[1] > max_length:
        context_ids = context_ids[:, -max_length:]

    prefix_len = min(prefix_ids.shape[1], context_ids.shape[1])
    draft_len = min(draft_ids.shape[1], max(context_ids.shape[1] - prefix_len, 0))

    embeddings = self.qwen.get_input_embeddings()
    context_embeds = embeddings(context_ids)

    if draft_len > 0:
        draft_start = prefix_len
        draft_end = min(draft_start + draft_len, context_embeds.shape[1])
        draft_embeds = context_embeds[:, draft_start:draft_end, :]
        projected = self.qwen.text_projector(draft_embeds.to(self.model_dtype)).to(context_embeds.dtype)

        before = context_embeds[:, :draft_start, :]
        after = context_embeds[:, draft_end:, :]
        context_embeds = torch.cat([before, projected, after], dim=1)

    output_ids = self.qwen.generate(
        inputs_embeds=context_embeds,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=self.tokenizer.pad_token_id,
        eos_token_id=self.tokenizer.eos_token_id,
    )
    gen_only = output_ids[:, context_embeds.shape[1]:]
    return self.tokenizer.decode(gen_only[0], skip_special_tokens=True).strip()

PlanProjectorTrainer.compute_loss = _compute_loss_no_inplace
PlanProjectorTrainer.generate_answer = _generate_answer_no_inplace
print('Patched PlanProjectorTrainer to avoid in-place ops in projector path.')

In [ ]:
# Hard-fix binding + grad sanity check for the current live model_wrapper instance.
import types

model_wrapper.compute_loss = types.MethodType(_compute_loss_no_inplace, model_wrapper)
model_wrapper.generate_answer = types.MethodType(_generate_answer_no_inplace, model_wrapper)

# Ensure only projector params are trainable (plus optional light tuning).
model_wrapper._configure_trainable_params(lightly_tune_qwen=LIGHTLY_TUNE_QWEN)

trainable = [p for p in model_wrapper.qwen.text_projector.parameters() if p.requires_grad]
print(f'Trainable projector tensors: {len(trainable)}')

# Check that loss tracks gradients before training loop.
with torch.enable_grad():
    probe = train_records[0]
    probe_loss = model_wrapper.compute_loss(
        question=probe['question'],
        draft_plan=probe['draft_plan'],
        gold_final=probe['gold_final'],
        max_length=MAX_LENGTH,
    )
print(f'Probe loss requires_grad: {probe_loss.requires_grad}')

In [ ]:
import torch

gpu_count = torch.cuda.device_count()
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Visible GPU count: {gpu_count}')
for i in range(gpu_count):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

if gpu_count < 1:
    raise RuntimeError('No GPU found. Enable GPU in Kaggle notebook settings.')

if gpu_count < 2:
    print('Warning: This notebook is configured for 2 GPUs; it will still run with a single GPU.')

In [ ]:
def train_projector(train_records):
    model_wrapper.qwen.train()
    optimizer = torch.optim.AdamW(
        model_wrapper.trainable_parameters(),
        lr=PROJECTOR_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    global_step = 0
    losses = []
    running_loss = 0.0
    no_grad_fallback_count = 0

    for epoch in range(NUM_TRAIN_EPOCHS):
        random.shuffle(train_records)
        optimizer.zero_grad(set_to_none=True)

        for idx, rec in enumerate(train_records, start=1):
            with torch.enable_grad():
                loss = model_wrapper.compute_loss(
                    question=rec['question'],
                    draft_plan=rec['draft_plan'],
                    gold_final=rec['gold_final'],
                    max_length=MAX_LENGTH,
                )

            # If the draft segment is fully truncated, loss can become disconnected from projector params.
            # Add a tiny projector L2 term so backward still has a valid gradient path.
            if not loss.requires_grad:
                reg = None
                for p in model_wrapper.qwen.text_projector.parameters():
                    term = (p.float() ** 2).mean()
                    reg = term if reg is None else (reg + term)
                loss = 1e-6 * reg
                no_grad_fallback_count += 1

            (loss / GRADIENT_ACCUMULATION_STEPS).backward()
            running_loss += loss.item()

            if idx % GRADIENT_ACCUMULATION_STEPS == 0 or idx == len(train_records):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                step_loss = running_loss / GRADIENT_ACCUMULATION_STEPS
                losses.append(step_loss)
                running_loss = 0.0

                if global_step % LOGGING_STEPS == 0:
                    print(f'epoch={epoch + 1} step={global_step} loss={step_loss:.6f}')

                if global_step % SAVE_STEPS == 0:
                    ckpt_dir = OUTPUT_ROOT / f'checkpoint-{global_step}'
                    ckpt_dir.mkdir(parents=True, exist_ok=True)
                    torch.save(model_wrapper.qwen.text_projector.state_dict(), ckpt_dir / 'text_projector.pth')
                    print(f'Saved projector checkpoint to: {ckpt_dir / "text_projector.pth"}')

    mean_loss = float(np.mean(losses)) if losses else 0.0
    return {
        'global_steps': global_step,
        'mean_loss': mean_loss,
        'no_grad_fallback_count': no_grad_fallback_count,
    }

print('Projector trainer ready.')
print(f'NUM_TRAIN_EPOCHS={NUM_TRAIN_EPOCHS}, projector_lr={PROJECTOR_LEARNING_RATE}, grad_accum={GRADIENT_ACCUMULATION_STEPS}')

In [ ]:
print('Starting projector training...')
train_stats = train_projector(train_records)

projector_path = FINAL_DIR / 'text_projector.pth'
torch.save(model_wrapper.qwen.text_projector.state_dict(), projector_path)
model_wrapper.tokenizer.save_pretrained(str(FINAL_DIR))

print('Training finished.')
print(f'Saved text projector to: {projector_path}')
print(train_stats)

In [ ]:
@torch.no_grad()
def _mcq_letter(text: str) -> str:
    m = re.search(r"\b([A-Da-d])\b", str(text).strip())
    return m.group(1).upper() if m else ''

def _compare_mcq(pred: str, gold: str) -> float:
    p = _mcq_letter(pred)
    g = _mcq_letter(gold)
    return float(p != '' and p == g)

def _extract_numeric_answer(text: str) -> str:
    s = str(text)
    boxed = re.findall(r'\\boxed\{([^}]*)\}', s)
    if boxed:
        s = boxed[-1]
    m = re.search(r"####\s*([-+]?[0-9][\d,\.]*)", s)
    if m:
        s = m.group(1)
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", s)
    if not nums:
        return ''
    return nums[-1].replace(',', '').strip()

def _compare_numeric(pred: str, gold: str) -> float:
    p = _extract_numeric_answer(pred)
    g = _extract_numeric_answer(gold)
    return float(p != '' and g != '' and p == g)

def _prepare_mmlu_sample(item: Dict) -> Dict[str, str]:
    q = item['question'].strip()
    c = item['choices']
    prompt = f"Question: {q}\nA. {c[0]}\nB. {c[1]}\nC. {c[2]}\nD. {c[3]}"
    ans = chr(ord('A') + int(item['answer']))
    return {'question': prompt, 'gold_final': ans}

def _prepare_gsm8k_sample(item: Dict) -> Dict[str, str]:
    return {'question': f"Question: {item['question']}", 'gold_final': str(item['answer'])}

def _prepare_aime_sample(item: Dict) -> Dict[str, str]:
    # Covers yentinglin/aime_2025 and HuggingFaceH4/aime_2024 schemas.
    problem = item.get('problem', item.get('Question', ''))
    answer = item.get('answer', item.get('Answer', ''))
    return {'question': f"Problem: {str(problem).strip()}", 'gold_final': str(answer).strip()}

def _load_benchmark_records(name: str, limit: int) -> List[Dict[str, str]]:
    if name == 'arc_easy':
        ds = load_dataset('allenai/ai2_arc', 'ARC-Easy', split='test').shuffle(seed=SEED)
        ds = ds.select(range(min(limit, len(ds))))
        return [prepare_arc_sample(x) for x in ds]
    if name == 'arc_challenge':
        ds = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test').shuffle(seed=SEED)
        ds = ds.select(range(min(limit, len(ds))))
        return [prepare_arc_sample(x) for x in ds]
    if name.startswith('dart-'):
        level = int(name.split('-')[-1])
        ds = load_dataset('hkust-nlp/dart-math-pool-math', split='train')
        ds = ds.filter(lambda x: x['query_metadata']['level'] == level)
        ds = ds.shuffle(seed=SEED)
        ds = ds.select(range(min(limit, len(ds))))
        return [prepare_dart_sample(x) for x in ds]
    if name == 'gsm8k':
        ds = load_dataset('gsm8k', 'main', split='test').shuffle(seed=SEED)
        ds = ds.select(range(min(limit, len(ds))))
        return [_prepare_gsm8k_sample(x) for x in ds]
    if name == 'mmlu':
        ds = load_dataset('cais/mmlu', 'all', split='test').shuffle(seed=SEED)
        ds = ds.select(range(min(limit, len(ds))))
        return [_prepare_mmlu_sample(x) for x in ds]
    if name == 'aime2025':
        ds = load_dataset('yentinglin/aime_2025', split='train').shuffle(seed=SEED)
        ds = ds.select(range(min(limit, len(ds))))
        return [_prepare_aime_sample(x) for x in ds]
    if name == 'aime2024':
        ds = load_dataset('HuggingFaceH4/aime_2024', split='train').shuffle(seed=SEED)
        ds = ds.select(range(min(limit, len(ds))))
        return [_prepare_aime_sample(x) for x in ds]
    raise ValueError(f'Unsupported benchmark: {name}')

def _run_benchmark_eval(name: str, records: List[Dict[str, str]]) -> Dict[str, float]:
    if name in {'arc_easy', 'arc_challenge', 'mmlu'}:
        compare_fn = _compare_mcq
    else:
        compare_fn = _compare_numeric

    correct = 0.0
    total = len(records)
    for ex in records:
        pred_text = model_wrapper.generate_answer(
            question=ex['question'],
            draft_plan=ex.get('draft_plan', 'Parse the quantities and solve step-by-step.'),
            max_length=MAX_LENGTH,
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
        )
        correct += compare_fn(pred_text, ex['gold_final'])

    accuracy = (correct / total) if total else 0.0
    return {'dataset': name, 'num_samples': total, 'accuracy': accuracy}

# Match eval_ours-style benchmark family.
ALL_BENCHMARKS = [
    'arc_easy',
    'arc_challenge',
    'dart-1',
    'dart-2',
    'dart-3',
    'dart-4',
    'dart-5',
    'gsm8k',
    'mmlu',
    'aime2024',
    'aime2025',
]

# Set this lower for quick smoke tests, higher for fuller eval.
BENCHMARK_NUM_SAMPLES = 100

benchmark_results = []
for benchmark in ALL_BENCHMARKS:
    print(f'Evaluating {benchmark}...')
    records = _load_benchmark_records(benchmark, BENCHMARK_NUM_SAMPLES)
    result = _run_benchmark_eval(benchmark, records)
    benchmark_results.append(result)
    print(f"{benchmark}: acc={result['accuracy']:.3f} over {result['num_samples']} samples")

overall_acc = float(np.mean([r['accuracy'] for r in benchmark_results])) if benchmark_results else 0.0
print(f'Overall mean accuracy across benchmarks: {overall_acc:.3f}')

meta = {
    'total_train_samples': TOTAL_TRAIN_SAMPLES,
    'per_dataset_train_samples_before_cap': PER_DATASET_TRAIN_SAMPLES,
    'projector_bottleneck_dim': PROJECTOR_BOTTLENECK_DIM,
    'projector_dropout': PROJECTOR_DROPOUT,
    'projector_learning_rate': PROJECTOR_LEARNING_RATE,
    'num_train_records': len(train_records),
    'num_eval_records': len(eval_records),
    'lightly_tune_qwen': LIGHTLY_TUNE_QWEN,
    'train_global_steps': train_stats['global_steps'],
    'train_mean_loss': train_stats['mean_loss'],
    'no_grad_fallback_count': train_stats.get('no_grad_fallback_count', 0),
    'benchmarks': benchmark_results,
    'overall_benchmark_mean_accuracy': overall_acc,
}

with open(OUTPUT_ROOT / 'training_meta.json', 'w', encoding='utf-8') as f:
    import json
    json.dump(meta, f, indent=2)

with open(OUTPUT_ROOT / 'benchmark_results.json', 'w', encoding='utf-8') as f:
    import json
    json.dump({'results': benchmark_results, 'overall_mean_accuracy': overall_acc}, f, indent=2)

print(f"Meta saved to: {OUTPUT_ROOT / 'training_meta.json'}")
print(f"Benchmarks saved to: {OUTPUT_ROOT / 'benchmark_results.json'}")

In [ ]:
# Debug one example: show LLaDA draft plan and a projector-conditioned refined plan text.

@torch.no_grad()
def _projected_generate_text(question: str, draft_plan: str, prefix: str, suffix: str, max_length: int = MAX_LENGTH, max_new_tokens: int = 128) -> str:
    model_wrapper.qwen.eval()
    prefix_ids = model_wrapper._tokenize_no_special(prefix, max_length)
    draft_ids = model_wrapper._tokenize_no_special(draft_plan, max_length)
    suffix_ids = model_wrapper._tokenize_no_special(suffix, max_length)

    context_ids = torch.cat([prefix_ids, draft_ids, suffix_ids], dim=1)
    if context_ids.shape[1] > max_length:
        context_ids = context_ids[:, -max_length:]

    prefix_len = min(prefix_ids.shape[1], context_ids.shape[1])
    draft_len = min(draft_ids.shape[1], max(context_ids.shape[1] - prefix_len, 0))

    embeddings = model_wrapper.qwen.get_input_embeddings()
    context_embeds = embeddings(context_ids)

    if draft_len > 0:
        draft_start = prefix_len
        draft_end = min(draft_start + draft_len, context_embeds.shape[1])
        draft_embeds = context_embeds[:, draft_start:draft_end, :]
        projected = model_wrapper.qwen.text_projector(draft_embeds.to(model_wrapper.model_dtype)).to(context_embeds.dtype)
        before = context_embeds[:, :draft_start, :]
        after = context_embeds[:, draft_end:, :]
        context_embeds = torch.cat([before, projected, after], dim=1)

    out_ids = model_wrapper.qwen.generate(
        inputs_embeds=context_embeds,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=model_wrapper.tokenizer.pad_token_id,
        eos_token_id=model_wrapper.tokenizer.eos_token_id,
    )
    gen_only = out_ids[:, context_embeds.shape[1]:]
    return model_wrapper.tokenizer.decode(gen_only[0], skip_special_tokens=True).strip()

# 1) Load one ARC-Easy sample.
arc_one = load_dataset('allenai/ai2_arc', 'ARC-Easy', split='test').shuffle(seed=SEED).select(range(1))[0]
arc_one_prepared = prepare_arc_sample(arc_one)

# 2) Generate one LLaDA draft plan for that sample.
print('Loading temporary LLaDA generator for one-sample debug...')
draft_generator_dbg = build_draft_generator()
llada_draft_plan = generate_draft_plan(arc_one_prepared['question'], draft_generator_dbg)
del draft_generator_dbg
free_cuda_memory()

# 3) Generate one projector-conditioned refined plan text.
refine_prefix = (
    'You are a careful planning assistant.\n'
    'Refine the draft plan into a clearer, shorter plan.\n'
    'Do not output the final answer letter or numeric result.\n\n'
    f"Question: {arc_one_prepared['question']}\n\n"
    'Draft plan:\n'
 )
refine_suffix = '\n\nRefined plan:\n'

refined_plan_text = _projected_generate_text(
    question=arc_one_prepared['question'],
    draft_plan=llada_draft_plan,
    prefix=refine_prefix,
    suffix=refine_suffix,
    max_length=MAX_LENGTH,
    max_new_tokens=128,
 )

print('\n===== ONE-EXAMPLE DEBUG (ARC-Easy) =====')
print('\nQuestion:\n', arc_one_prepared['question'])
print('\nLLaDA draft plan:\n', llada_draft_plan)
print('\nProjector-conditioned refined plan (text view):\n', refined_plan_text)